In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler


In [4]:
path = r"C:\Users\vikas\.a_DS_Data_Folder\Customer-Churn.csv"
data = pd.read_csv(path)
data.sample(1)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
3910,8938-UMKPI,Female,0,No,No,47,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,106.4,5127.95,Yes


In [5]:
pd.DataFrame(data.iloc[2456])

,2456
customerID,5968-HYJRZ
gender,Male
SeniorCitizen,0
Partner,Yes
Dependents,Yes
tenure,47
PhoneService,Yes
MultipleLines,Yes
InternetService,No
OnlineSecurity,No internet service


In [6]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
data.shape

(7043, 21)

In [9]:
data.isnull().sum()
data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)

C:\Users\vikas\AppData\Local\Temp\ipykernel_6728\4280634192.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)


In [10]:
data.sample(1)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
2747,8295-FHIVV,Male,0,No,No,7,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,19.4,168.65,No


In [11]:
sns.boxplot(data['MonthlyCharges'], orient='h')
plt.show()

NameError: name 'sns' is not defined

**Cheacking corelation numeric vs target & categories vs target**

## cheacking numeric correlation with target variable(churn)

In [12]:
# Remove column spaces
data.columns = data.columns.str.strip()

# Convert target to numeric
data['Churn'] = data['Churn'].map({'Yes':1, 'No':0})

# Check correlation
data.corr(numeric_only=True)['Churn'].sort_values()


tenure           -0.352229
TotalCharges     -0.199037
SeniorCitizen     0.150889
MonthlyCharges    0.193356
Churn             1.000000
Name: Churn, dtype: float64

In [13]:
cat = data.select_dtypes(include = 'object').columns
num = data.select_dtypes(exclude = 'object').columns
print("Categorical Columns :\n",cat)
print("Numerical Columns :\n",num)

Categorical Columns :
 Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')
Numerical Columns :
 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'], dtype='object')


In [14]:
cat

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

## If P-value < 0.05 → Feature is correlated with target
## If P-value > 0.05 → Feature is NOT correlated
## Checking correlation categorical column with target column

In [15]:
cat

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

## cheacking categoric correlation with target variable(churn)

In [16]:
table = pd.crosstab(data['customerID'], data['Churn'])
chi2, p, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['gender'], data['Churn'])
chi2, p1, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['Partner'], data['Churn'])
chi2, p2, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['Dependents'], data['Churn'])
chi2, p3, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['MultipleLines'], data['Churn'])
chi2, p4, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['InternetService'], data['Churn'])
chi2, p5, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['OnlineSecurity'], data['Churn'])
chi2, p6, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['DeviceProtection'], data['Churn'])
chi2, p7, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['TechSupport'], data['Churn'])
chi2, p8, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['StreamingTV'], data['Churn'])
chi2, p9, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['PaperlessBilling'], data['Churn'])
chi2, p10, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['StreamingMovies'], data['Churn'])
chi2, p11, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['PaymentMethod'], data['Churn'])
chi2, p12, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['Contract'], data['Churn'])
chi2, p13, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['PhoneService'], data['Churn'])
chi2, p14, dof, expected = chi2_contingency(table)

table = pd.crosstab(data['OnlineBackup'], data['Churn'])
chi2, p15, dof, expected = chi2_contingency(table)



print("P-value of custemerid:", p)
print("P-value of gender:", p1)
print("P-value of Partner:", p2)
print("P-value of Dependents:", p3)
print("P-value of MultipleLines:", p4)
print("P-value of InternetService:", p5)
print("P-value of OnlineSecurity:", p6)
print("P-value of DeviceProtection:", p7)
print("P-value of TechSupport:", p8)
print("P-value of StreamingTV:", p9)
print("P-value of PaperlessBilling:", p10)
print("P-value of StreamingMovies:", p11)
print("P-value of PaymentMethod:", p12)
print("P-value of Contract:", p13)
print("P-value of PhoneService:", p14)
print("P-value of OnlineBackup:", p15)

NameError: name 'chi2_contingency' is not defined

In [17]:
data.drop(['customerID', 'gender','PhoneService'], axis=1, inplace=True)

In [18]:
data

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Yes,No,1,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,0,No,No,34,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,0,No,No,2,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,0,No,No,45,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,0,No,No,2,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,Yes,Yes,24,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7039,0,Yes,Yes,72,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7040,0,Yes,Yes,11,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0
7041,1,Yes,No,4,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1


## cheacking columns with more than 2 categories or not for fix label encoder and one hot encoder 

In [19]:
data['StreamingTV'].value_counts()

StreamingTV
No                     2810
Yes                    2707
No internet service    1526
Name: count, dtype: int64

## Label_Encoder

In [20]:
scaler = LabelEncoder()
data['Partner'] = scaler.fit_transform(data['Partner'])
data['Dependents'] = scaler.fit_transform(data['Dependents'])
data['StreamingTV'] = scaler.fit_transform(data['StreamingTV'])
data['PaperlessBilling'] = scaler.fit_transform(data['PaperlessBilling'])
data['DeviceProtection'] = scaler.fit_transform(data['DeviceProtection'])
data['OnlineSecurity'] = scaler.fit_transform(data['OnlineSecurity'])
data['OnlineBackup'] = scaler.fit_transform(data['OnlineBackup'])
data['MultipleLines'] = scaler.fit_transform(data['MultipleLines'])
data['TechSupport'] = scaler.fit_transform(data['TechSupport'])
data['StreamingMovies'] = scaler.fit_transform(data['StreamingMovies'])


In [21]:
data.sample(5)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
2102,0,0,1,18,2,Fiber optic,0,0,2,0,0,0,Month-to-month,1,Credit card (automatic),80.55,1406.65,0
288,1,0,0,8,0,Fiber optic,0,2,0,0,0,0,Month-to-month,1,Electronic check,74.50,606.55,1
2424,0,0,0,57,2,Fiber optic,2,2,2,0,2,2,Two year,1,Bank transfer (automatic),112.95,6465.00,1
1023,1,1,0,45,0,Fiber optic,0,0,2,0,0,2,Month-to-month,1,Electronic check,86.10,3861.45,0
3210,0,1,1,64,0,DSL,2,0,0,2,2,0,One year,0,Mailed check,65.80,4068.00,0


In [22]:
data['InternetService'].value_counts()

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

## string converting to numeric and droping column (Feature selection)

In [23]:

data['Contract_Duration'] = data['Contract'].apply(
    lambda x: 0 if x == 'Month-to-month'
    else 1 if x == 'One year'
    else 2
)
data['PaymentMethod'] = data['PaymentMethod'].apply(
    lambda x: 0 if x == 'Electronic check'
    else 1 if x == 'Mailed check'
    else 2 if x == 'Bank transfer(automatic)'
    else 3 if x == 'Credit card (automatic)'
    else 4
)
data['InternetService'] = data['InternetService'].apply(
    lambda x: 0 if x == 'Fiber optic'
    else 1 if x == 'DSL'
    else 2 if x == 'No'
    else 3
)


data.drop(['Contract'], axis=1, inplace=True)

data.sample(1)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Contract_Duration
5939,0,0,0,18,1,1,2,0,2,0,0,0,0,1,33.6,550.35,0,0


## Scaling numeric data otherwise model will think big values are more important than others

In [24]:
data.sample(2)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Contract_Duration
5497,0,1,1,49,1,1,0,2,0,0,2,2,1,4,51.80,2541.25,1,1
4802,0,0,0,28,0,1,0,2,0,2,0,0,0,4,54.65,1517.50,0,0


In [25]:
standard = StandardScaler()
data[['TotalCharges','MonthlyCharges']] = standard.fit_transform(data[['TotalCharges','MonthlyCharges']])

In [26]:
data.sample(2)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Contract_Duration
2796,0,1,0,55,2,0,2,0,0,0,2,0,1,4,0.832172,1.182143,0,0
2747,0,0,0,7,0,2,1,1,1,1,1,1,1,1,-1.507638,-0.932965,0,0


## separeting the data and lebels

In [27]:
X = data.drop(columns = 'Churn',axis = 1)
y = data['Churn']
print(X,y)

      SeniorCitizen  Partner  Dependents  tenure  MultipleLines  \
0                 0        1           0       1              1   
1                 0        0           0      34              0   
2                 0        0           0       2              0   
3                 0        0           0      45              1   
4                 0        0           0       2              0   
...             ...      ...         ...     ...            ...   
7038              0        1           1      24              2   
7039              0        1           1      72              2   
7040              0        1           1      11              1   
7041              1        1           0       4              2   
7042              0        0           0      66              0   

      InternetService  OnlineSecurity  OnlineBackup  DeviceProtection  \
0                   1               0             2                 0   
1                   1               2            

In [28]:
X_train,X_test,y_train,y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

**1️⃣ Data is not scaled properly
2️⃣ Dataset is complex
3️⃣ Default iterations (100) are too small
✅ For Best Solutions
Increase Iterations (Most Common Fix)**

In [29]:
lr_model = LogisticRegression(class_weight='balanced',max_iter=1000)
lr_model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

## Accuracy

In [30]:
y_pred = lr_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Your Accuracy of model is :",accuracy)

Your Accuracy of model is : 0.759403832505323


In [31]:
print("Train Accuracy:", model.score(X_train, y_train))
print("Test Accuracy:", model.score(X_test, y_test))
print("model accuracy Difference = 74.56 − 75.94 ≈ 1.38% . so it's not overfitted ")

NameError: name 'model' is not defined

In [824]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Report:\n", classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred))


Accuracy: 0.759403832505323
Confusion Matrix:
 [[757 279]
 [ 60 313]]
Report:
               precision    recall  f1-score   support

           0       0.93      0.73      0.82      1036
           1       0.53      0.84      0.65       373

    accuracy                           0.76      1409
   macro avg       0.73      0.78      0.73      1409
weighted avg       0.82      0.76      0.77      1409

ROC AUC: 0.7849185359238978


## Now Using Random Forest model

In [825]:
X_train1,X_test1,y_train1,y_test1 = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [826]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier()
rf_model.fit(X_train1,y_train1)

RandomForestClassifier()

In [827]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

y_pred1 = rf_model.predict(X_test1)
print("Accuracy:", accuracy_score(y_test1, y_pred1))
print("Confusion Matrix:\n", confusion_matrix(y_test1, y_pred1))
print("Report:\n", classification_report(y_test1, y_pred1))
print("ROC AUC:", roc_auc_score(y_test1, y_pred1))


Accuracy: 0.7892122072391767
Confusion Matrix:
 [[940  96]
 [201 172]]
Report:
               precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.64      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409

ROC AUC: 0.6842309563489187


## Now Using Gradient Boosting model

In [828]:
X_train2,X_test2,y_train2,y_test2 = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [829]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier()
gb_model.fit(X_train2,y_train2)

GradientBoostingClassifier()

In [830]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

y_pred2 = gb_model.predict(X_test2)
print("Accuracy:", accuracy_score(y_test2, y_pred2))
print("Confusion Matrix:\n", confusion_matrix(y_test2, y_pred2))
print("Report:\n", classification_report(y_test2, y_pred2))
print("ROC AUC:", roc_auc_score(y_test2, y_pred2))


Accuracy: 0.8140525195173882
Confusion Matrix:
 [[940  96]
 [166 207]]
Report:
               precision    recall  f1-score   support

           0       0.85      0.91      0.88      1036
           1       0.68      0.55      0.61       373

    accuracy                           0.81      1409
   macro avg       0.77      0.73      0.75      1409
weighted avg       0.81      0.81      0.81      1409

ROC AUC: 0.7311478464293478


## Now Using XGBoost model

In [831]:
X_train3,X_test3,y_train3,y_test3 = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [832]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier()
xgb_model.fit(X_train3,y_train3)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [833]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

y_pred3 = xgb_model.predict(X_test3)
print("Accuracy:", accuracy_score(y_test3, y_pred3))
print("Confusion Matrix:\n", confusion_matrix(y_test3, y_pred3))
print("Report:\n", classification_report(y_test3, y_pred3))
print("ROC AUC:", roc_auc_score(y_test3, y_pred3))


Accuracy: 0.7920511000709723
Confusion Matrix:
 [[920 116]
 [177 196]]
Report:
               precision    recall  f1-score   support

           0       0.84      0.89      0.86      1036
           1       0.63      0.53      0.57       373

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

ROC AUC: 0.7067500284658461


## ⭐ Step 1: Understand Metrics Meaning
## ✅ Accuracy

## 👉 Overall correct predictions

## ✅ ROC-AUC

## 👉 How well model separates churn vs non-churn
## 👉 More important for churn datasets (imbalanced data)

In [834]:
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

results = []

models = {
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "XGBoost": xgb_model,
    "Logistic Regression": lr_model
}

for name, model in models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    results.append([name, acc, roc])

comparison_df = pd.DataFrame(results, columns=["Model","Accuracy","ROC AUC"])
print(comparison_df)


                 Model  Accuracy   ROC AUC
0        Random Forest  0.789212  0.829041
1    Gradient Boosting  0.814053  0.859068
2              XGBoost  0.792051  0.837915
3  Logistic Regression  0.759404  0.861679


## Now We Get Logistic Regression is perfect and good Evaluation we goted

In [ ]:
import pandas as pd

# ---------- TAKE USER INPUT ----------
input_data = {
    "SeniorCitizen": int(input("SeniorCitizen (0 = No, 1 = Yes): ")),

    "Partner": input("Partner (Yes/No): "),
    "Dependents": input("Dependents (Yes/No): "),

    "tenure": int(input("Tenure (months): ")),

    "MultipleLines": input("MultipleLines (Yes/No/No phone service): "),
    "InternetService": input("InternetService (DSL/Fiber optic/No): "),

    "OnlineSecurity": input("OnlineSecurity (Yes/No/No internet service): "),
    "OnlineBackup": input("OnlineBackup (Yes/No/No internet service): "),
    "DeviceProtection": input("DeviceProtection (Yes/No/No internet service): "),
    "TechSupport": input("TechSupport (Yes/No/No internet service): "),

    "StreamingTV": input("StreamingTV (Yes/No/No internet service): "),
    "StreamingMovies": input("StreamingMovies (Yes/No/No internet service): "),

    "PaperlessBilling": input("PaperlessBilling (Yes/No): "),

    "PaymentMethod": input(
        "PaymentMethod (Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic)): "
    ),

    "MonthlyCharges": float(input("MonthlyCharges: ")),
    "TotalCharges": float(input("TotalCharges: ")),

    "Contract_Duration": int(input("Contract Duration (0 = Month-to-month, 1 = One year, 2 = Two year): "))
}

# ---------- CONVERT TO DATAFRAME ----------
user_df = pd.DataFrame([input_data])

# ---------- SAME ENCODING USED IN TRAINING ----------
user_df = pd.get_dummies(user_df)

# ---------- Match Training Columns ----------
user_df = user_df.reindex(columns=X_train.columns, fill_value=0)

# ---------- PREDICT ----------
prediction = lr_model.predict(user_df)


# ---------- RESULT ----------
if prediction[0] == 1:
    print("Customer will Churn ")
else:
    print("Customer will Stay ")


SeniorCitizen (0 = No, 1 = Yes):  0
Partner (Yes/No):  yes
Dependents (Yes/No):  yes
Tenure (months):  47
MultipleLines (Yes/No/No phone service):  yes
InternetService (DSL/Fiber optic/No):  no
OnlineSecurity (Yes/No/No internet service):  /No internet service
OnlineBackup (Yes/No/No internet service):  /No internet service
DeviceProtection (Yes/No/No internet service):  /No internet service
TechSupport (Yes/No/No internet service):  No internet service
StreamingTV (Yes/No/No internet service):  No internet service
StreamingMovies (Yes/No/No internet service):  No internet service
PaperlessBilling (Yes/No):  no
PaymentMethod (Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic)):  Mailed check
MonthlyCharges:  24.55
TotalCharges:  1160.45


In [841]:
pd.DataFrame(X)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Contract_Duration
0,0,1,0,1,1,1,0,2,0,0,0,0,1,0,-1.160323,-0.994242,0
1,0,0,0,34,0,1,2,0,2,0,0,0,0,1,-0.259629,-0.173244,1
2,0,0,0,2,0,1,2,2,0,0,0,0,1,1,-0.362660,-0.959674,0
3,0,0,0,45,1,1,2,0,2,2,0,0,0,4,-0.746535,-0.194766,1
4,0,0,0,2,0,0,0,0,0,0,0,0,1,0,0.197365,-0.940470,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,1,1,24,2,1,2,0,2,2,2,2,1,1,0.665992,-0.128655,1
7039,0,1,1,72,2,0,0,2,2,0,2,2,1,3,1.277533,2.243151,1
7040,0,1,1,11,1,1,2,0,0,0,0,0,1,0,-1.168632,-0.854469,0
7041,1,1,0,4,2,0,0,0,0,0,0,0,1,1,0.320338,-0.872062,0
